# Continued Pre-Training (CPT) of LLM with PyTorch FSDP and QLora on Amazon SageMaker Training jobs

## Lab 1 - Data preparation

In this notebook, we are going to prepare the dataset for later on customize Qwen/Qwen3-4B-Instruct-2507

## Prerequisites

### Install requirements

In [ ]:
%pip install -r ./scripts/requirements.txt --upgrade
%pip install s3fs

### Setup and dependencies

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

***

## Visualize and upload the dataset

We are going to load [FreedomIntelligence/medical-o1-reasoning-SFT](https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT) dataset

In [ ]:
import datasets
from datasets import load_dataset

dataset = (
    load_dataset(
        "FreedomIntelligence/medical-o1-reasoning-SFT",
        "en",
        split="train",
        streaming=True,
    )
    .take(1000)
    .shuffle(buffer_size=500)
)

dataset = datasets.Dataset.from_generator(lambda: dataset, features=dataset.features)

dataset

With this function, we are going to tokenize text input into fixed blocks

In [ ]:
from itertools import chain

def group_texts(examples, block_size=2048):
    """
    Groups a list of tokenized text examples into fixed-size blocks for language model training.

    Args:
        examples (dict): A dictionary where keys are feature names (e.g., "input_ids") and values 
                         are lists of tokenized sequences.
        block_size (int, optional): The size of each chunk. Defaults to 2048.

    Returns:
        dict: A dictionary containing the grouped chunks for each feature. An additional "labels" key 
              is included, which is a copy of the "input_ids" key.
    """
    # Concatenate all texts.
    concatenated_examples = {k: list(chain(*examples[k])) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of max_len.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

Load the Tokenizer of the model that will be used for the Continued pre-training job

In [ ]:
from transformers import AutoTokenizer

model_id = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
from functools import partial

column_names = dataset.column_names

train_dataset = dataset.map(
    lambda sample: tokenizer(sample["text"], return_token_type_ids=False),
    batched=True,
    remove_columns=list(column_names),
).map(
    partial(group_texts, block_size=2048),
    batched=True,
)

### Upload to Amazon S3

In [ ]:
# save train_dataset to s3 using our SageMaker session
if default_prefix:
    input_path = f'{default_prefix}/datasets/llm-fine-tuning-cpt'
else:
    input_path = f'datasets/llm-fine-tuning-cpt'

train_dataset.save_to_disk(f"s3://{bucket_name}/{input_path}/train")
train_dataset_s3_path = f"s3://{bucket_name}/{input_path}/train"

print(f"Training data uploaded to:")
print(train_dataset_s3_path)